# Can Jev recover suicide cases missed by XGBoost?

This notebook reports a prespecified error study using the original centralized Word2Vec + XGBoost pipeline. It prepares **100 XGBoost mistakes**: **40 training**, **20 validation**, and **40 held-out evaluation** cases, balanced between false negatives and false positives. The primary outcome is recovery of the 20 held-out suicide false negatives. False-positive controls detect the limitation of an always-positive answer.

**Weight fine-tuning is blocked and was not run.** The repository exposes only Jev inference. Authenticated conventional OpenAPI routes returned HTTP 404, official training documentation was inaccessible, and no separate training-fee cap was available. Prepared training records are not a trained model. No prompt optimization is presented as weight training.

The default Run All reads committed, redacted results and makes **no paid requests**. Raw texts, training records, model caches, credentials, and the durable budget ledger remain local and ignored by Git.

In [1]:
from pathlib import Path
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "jev_research").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "research/results/jev_error_study"
ARTIFACTS = ROOT / "artifacts/jev_study"
RUN_BASELINE_PREPARATION = False
RUN_PAID_BASE_INFERENCE = False
assert sys.version_info[:2] == (3, 12), "Use the pinned Python 3.12 environment."
print("Default mode: saved-results review; paid inference disabled.")

Default mode: saved-results review; paid inference disabled.


## Protocol and leakage controls

- Reference: the original **centralized** model, seed 42, original cleaning and 64%/16%/20% stratified split; XGBoost threshold fixed at **0.50**. The study does not select among the nine notebook configurations after seeing errors.
- Jev training/validation cases come only from XGBoost's validation errors. The final Jev evaluation uses XGBoost's test errors. Exact raw and cleaned texts are disjoint across those source splits.
- Hash-ranked selection is fixed before Jev calls. Each role has equal FN/FP counts. This artificial mixture supports separate conditional error analyses, not natural-prevalence accuracy.
- Inference payloads include text and the fixed scoring question, without gold labels or error metadata. Jev scores use threshold **0.50**.
- One SQLite ledger caps the entire study at **100 paid inference attempts and 100 unique samples**. Reservations precede network I/O; failures and interruptions count, retries/redirects are disabled, and exact successful responses are cached. Keep this ledger: deleting it resets local accounting.
- Base evaluation requires 40 attempts. A future supported tuned-model comparison on the same 40 cases would require another 40, leaving 20 attempts unused. Training fees are separate and have not been authorized.

In [2]:
import os
if RUN_BASELINE_PREPARATION:
    from jev_research.study import prepare_study
    csv_path = os.environ.get("SUICIDE_DATA_PATH")
    if not csv_path:
        raise RuntimeError("Set SUICIDE_DATA_PATH to the original Suicide_Detection.csv.")
    prepare_study(csv_path, ARTIFACTS, RESULTS)

study = json.loads((RESULTS / "study.json").read_text())
training_status = json.loads((RESULTS / "training_status.json").read_text())
print("Dataset SHA-256:", study["dataset_sha256"])
print("Training status:", training_status["status"])
print("Prepared training roles:", study["training_counts"])
print("Held-out evaluation count:", study["evaluation_count"])
print("Full XGBoost test metrics:", study["full_xgboost_test_metrics"])

Dataset SHA-256: 1ec16b065a2fae36488de16e8abfc326586a576a25901e09467223e545ddd295
Training status: blocked_not_run
Prepared training roles: {'train': 40, 'validation': 20}
Held-out evaluation count: 40
Full XGBoost test metrics: {'accuracy': 0.9196840878684562, 'fn': 1835, 'fp': 1887, 'n': 46342, 'positive_f1': 0.9198639280024115, 'roc_auc': 0.9751687763872223, 'threshold': 0.5, 'tn': 21258, 'tp': 21362}


In [3]:
import pandas as pd
manifest = pd.read_json(RESULTS / "selection_manifest.json")
assert len(manifest) == 100 and manifest.sample_id.nunique() == 100
assert manifest.jev_role.value_counts().to_dict() == {"train": 40, "evaluation": 40, "validation": 20}
assert manifest.loc[manifest.jev_role.eq("evaluation"), "source_split"].eq("test").all()
assert manifest.loc[~manifest.jev_role.eq("evaluation"), "source_split"].eq("validation").all()
assert manifest.y_true.ne(manifest.y_xgb).all()
assert not {"text", "cleaned_text"}.intersection(manifest.columns)
display(pd.crosstab(manifest.jev_role, manifest.error_type).reindex(["train", "validation", "evaluation"]))

error_type,false_negative,false_positive
jev_role,,
train,20,20
validation,10,10
evaluation,20,20


## Optional paid execution

Enable the next flag only when intentionally spending the remaining study budget. Set `METIS_API_KEY` securely in the process environment. `main.py` now dispatches the guarded CLI and sends no unbudgeted demonstration requests. Existing successful results are read from the ledger without spending again. A provider error stops execution for diagnosis; this notebook does not retry it automatically.

The private cohort and original ledger are required to resume a study. A fresh clone can review the published evidence without those raw artifacts; it should not recreate the ledger to repeat this study unnoticed.

In [4]:
if RUN_PAID_BASE_INFERENCE:
    from jev_research.study import run_base_evaluation
    run_base_evaluation(ARTIFACTS, RESULTS)

budget = json.loads((RESULTS / "inference_budget.json").read_text())
assert budget["attempts_spent"] <= 100
assert budget["unique_samples_attempted"] <= 100
print({key: budget[key] for key in ["max_attempts", "attempts_spent", "remaining_attempts", "succeeded", "failed", "pending_uncertain"]})

{'max_attempts': 100, 'attempts_spent': 40, 'remaining_attempts': 60, 'succeeded': 40, 'failed': 0, 'pending_uncertain': 0}


## Observed results

The base-model results below are measured on held-out XGBoost mistakes. Wilson 95% intervals describe binomial uncertainty conditional on these sampled errors. The fine-tuned arm has no results because weight training was not possible. No paired benefit or statistical significance can be estimated without that arm.

In [5]:
from IPython.display import Markdown, display
report = json.loads((RESULTS / "report.json").read_text())
assert report["metadata"]["training_status"] == "blocked_not_run"
assert report["primary_paired"]["all_errors"]["paired_n"] == 0
display(Markdown((RESULTS / "report.md").read_text()))

# Jev correction of held-out XGBoost errors

Selected errors: 40; complete paired responses: 0; unresolved pairs: 40.

The prespecified score threshold is 0.50.

## Available model responses

These rates use each model's available responses. All selected denominators and unresolved outcomes remain explicit; rates from different response subsets are not a paired comparison.

| Outcome | Base Jev | Fine-tuned Jev (not run) |
| --- | --- | --- |
| Missed-suicide recovery (XGBoost FN) | 11/20 (55.0%; 95% Wilson CI 34.2%–74.2%); selected n=20 | Not estimable (n=0); selected n=20 |
| False-alarm correction (XGBoost FP) | 15/20 (75.0%; 95% Wilson CI 53.1%–88.8%); selected n=20 | Not estimable (n=0); selected n=20 |
| All sampled errors | 26/40 (65.0%; 95% Wilson CI 49.5%–77.9%); selected n=40 | Not estimable (n=0); selected n=40 |

Weight fine-tuning was not run. No tuned-model behavior or benefit has been established.

## Primary paired comparison

Rates below use the same complete response pairs for both models.

| Outcome | Base Jev | Fine-tuned Jev (not run) | Paired / selected |
| --- | --- | --- | --- |
| Missed-suicide recovery (XGBoost FN) | Not estimable (n=0) | Not estimable (n=0) | 0 / 20 |
| False-alarm correction (XGBoost FP) | Not estimable (n=0) | Not estimable (n=0) | 0 / 20 |
| All sampled errors | Not estimable (n=0) | Not estimable (n=0) | 0 / 40 |

No complete response pairs are available. Model improvement, regression, and paired statistical significance cannot be estimated.

## Unresolved responses and all-selected bounds

| Model | Observed / selected | Failed | Missing | Corrected | Possible all-selected correction rate |
| --- | --- | --- | --- | --- | --- |
| Base Jev | 40 / 40 | 0 | 0 | 26 | 65.0%–65.0% |
| Fine-tuned Jev (not run) | 0 / 40 | 0 | 40 | 0 | 0.0%–100.0% |

These bounds assign every unresolved outcome first to incorrect, then to correct. They are missing-outcome bounds, not statistical confidence intervals.

## Interpretation limits

- Gold labels select XGBoost errors; this is oracle-selected error analysis, not a deployable cascade evaluation.
- Correction rates are conditional on the sampled error mixture; deliberate FN/FP balancing does not estimate natural prevalence.
- The primary comparison uses complete response pairs. Missing and failed outcomes remain explicit, with worst/best bounds over all selected samples.
- Wilson 95% intervals describe binomial uncertainty within the sampled errors; small stratum counts produce wide intervals.
- Social-media dataset labels do not establish clinical validity or deployment performance.
- The original notebook inspected the source test split, and base outcomes are now public. A later tuned comparison reuses inspected cases; freeze all tuning choices on training/validation and describe it as exploratory.
- jev-latest is a mutable provider alias without a recorded immutable revision; later backend changes may confound comparisons performed at different times.


In [6]:
import matplotlib.pyplot as plt
import numpy as np

names = ["false_negative", "false_positive"]
labels = ["Missed-suicide recovery", "False-alarm correction"]
observed = [report["all_selected_by_arm"]["base"][name]["observed"] for name in names]
rates = np.array([item["rate"] for item in observed], dtype=float)
intervals = np.array([item["wilson_95"] for item in observed], dtype=float)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(labels, rates, color=["#2563eb", "#0d9488"], width=0.55)
ax.errorbar(labels, rates, yerr=np.vstack([rates - intervals[:, 0], intervals[:, 1] - rates]),
            fmt="none", ecolor="black", capsize=6)
for index, item in enumerate(observed):
    ax.text(index, min(1.04, rates[index] + 0.04), f"{item['corrected']}/{item['n']}", ha="center")
ax.set(ylim=(0, 1.12), ylabel="Conditional correction rate",
       title="Base Jev on held-out XGBoost errors (95% Wilson intervals)")
fig.tight_layout()
(ARTIFACTS / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ARTIFACTS / "figures/base_correction_rates.png", dpi=180)
plt.show()
plt.close(fig)

## What remains for true weight fine-tuning

1. Obtain the provider's supported Jev training/upload/job API and confirm how a resulting custom model is invoked through inference.
2. Establish training access and approve a separate maximum training spend. The 100-attempt inference limit does not authorize training fees.
3. Convert the ignored `train.jsonl` (40 rows) and `validation.jsonl` (20 rows) to that documented schema. Keep all 40 evaluation rows out of training and model selection.
4. Record the training job ID, base/custom model identifiers, exact training manifest hashes, training configuration, completed status, and actual spend.
5. Evaluate the resulting model on the same 40 held-out errors using the same durable budget ledger, then produce the paired report. Do not compare a fabricated or prompt-only model under a fine-tuned label.

This experiment is **oracle-selected error analysis** because gold labels identify XGBoost mistakes. It cannot establish the performance of an operational cascade, overall recall gains, clinical validity, or specificity on the original population. The original social-media labels and the small held-out sample impose additional limits.

The original notebook previously inspected the XGBoost test split, and this report now publishes the selected cases and base outcomes. A later tuned comparison on these cases is exploratory: freeze training choices using training/validation only. The remote alias `jev-latest` has no recorded immutable backend revision, so later backend changes can confound a comparison performed at a different time.